# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

**Progression : exercices 1.1, 2.1, 3.1 et 3.2 réalisés.**

In [1]:
import json
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName('J2_Spark_Delta_Silver')
         .master('local[2]').config('spark.sql.shuffle.partitions', '8').getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
REPORT = {'scope': 'Exercices 1.1 et 2.1 seulement', 'spark_version': spark.version}
print('Session active :', spark.version, '| Spark UI : http://localhost:4040')

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c159eca4-ef78-4bbd-a6f8-84f5516f24f9;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central


	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central


	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central


	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 868ms :: artifacts dl 37ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1

	0 artifacts copied, 33 already retrieved (0kB/43ms)


26/11/19 10:32:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Session active : 4.2.0 | Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [2]:
BRONZE_TABLES = {name: f'/lakehouse/bronze/{name}' for name in ['flights', 'airlines', 'airports']}
bronze_dfs = {name: spark.read.format('delta').load(path) for name, path in BRONZE_TABLES.items()}
df_bronze = bronze_dfs['flights']
REPORT['bronze_rows'] = {}
REPORT['missing_values'] = {}
for name, df in bronze_dfs.items():
    REPORT['bronze_rows'][name] = df.count()
    cols = [c for c in df.columns if not c.startswith('_')]
    # Les chaînes vides ou constituées d'espaces sont aussi considérées manquantes.
    null_report = df.agg(*[
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast('string')) == ''), 1).otherwise(0)).alias(c)
        for c in cols])
    REPORT['missing_values'][name] = null_report.first().asDict()
    print(name, ':', REPORT['bronze_rows'][name], 'lignes ; manquants :')
    null_report.show(truncate=False)

business_cols = [c for c in df_bronze.columns if not c.startswith('_')]
# Doublons stricts métier : ignorer les métadonnées d'ingestion, conserver les vols distincts.
flights_unique = df_bronze.dropDuplicates(business_cols).cache()
REPORT['strict_duplicates_removed'] = REPORT['bronze_rows']['flights'] - flights_unique.count()
print('Doublons stricts métier :', REPORT['strict_duplicates_removed'])
REPORT['dimension_keys'] = {}
for name in ['airlines', 'airports']:
    df = bronze_dfs[name].withColumn('iata_code', F.upper(F.trim('iata_code')))
    missing = df.filter(F.col('iata_code').isNull() | (F.col('iata_code') == '')).count()
    duplicate_keys = df.groupBy('iata_code').count().filter('count > 1').count()
    REPORT['dimension_keys'][name] = {'missing': missing, 'duplicate_keys': duplicate_keys}
    assert missing == 0 and duplicate_keys == 0, f'Clés ambiguës dans {name} : corriger avant jointure.'
print('Clés des dimensions contrôlées :', REPORT['dimension_keys'])

flights : 1943809 lignes ; manquants :


+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|year|month|day|day_of_week|airline|flight_number|tail_number|origin_airport|destination_airport|scheduled_departure|departure_time|departure_delay|taxi_out|wheels_off|scheduled_time|elapsed_time|air_time|distance|wheels_on|taxi_in|scheduled_arrival|arrival_time|arrival_delay|diverted|cancelled|cancellation_reason|air_system_delay|security_delay|airline_delay|late_aircraft_delay|weather_delay|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

airlines : 14 lignes ; manquants :


+---------+-------+
|iata_code|airline|
+---------+-------+
|0        |0      |
+---------+-------+



airports : 322 lignes ; manquants :


+---------+-------+----+-----+-------+--------+---------+
|iata_code|airport|city|state|country|latitude|longitude|
+---------+-------+----+-----+-------+--------+---------+
|0        |0      |0   |0    |0      |3       |3        |
+---------+-------+----+-----+-------+--------+---------+



Doublons stricts métier : 0


Clés des dimensions contrôlées : {'airlines': {'missing': 0, 'duplicate_keys': 0}, 'airports': {'missing': 0, 'duplicate_keys': 0}}


## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

In [3]:
SILVER_MAIN_PATH = '/lakehouse/silver/flights'
integer_cols = ['year', 'month', 'day', 'day_of_week', 'flight_number', 'cancelled', 'diverted']
number_cols = ['departure_delay', 'arrival_delay', 'taxi_out', 'scheduled_time', 'elapsed_time',
               'air_time', 'distance', 'taxi_in', 'air_system_delay', 'security_delay',
               'airline_delay', 'late_aircraft_delay', 'weather_delay']
cast_types = {**{c: 'int' for c in integer_cols}, **{c: 'double' for c in number_cols}}
# TRY_CAST rend visibles les conversions invalides sans interrompre le diagnostic.
REPORT['invalid_numeric_casts'] = flights_unique.agg(*[
    F.sum(F.when(F.col(c).isNotNull() & (F.trim(F.col(c)) != '') &
                 F.expr(f'try_cast(`{c}` as {t})').isNull(), 1).otherwise(0)).alias(c)
    for c, t in cast_types.items()]).first().asDict()
clean = flights_unique
for c, dtype in cast_types.items():
    clean = clean.withColumn(c, F.expr(f'try_cast(`{c}` as {dtype})'))
for c in ['airline', 'origin_airport', 'destination_airport']:
    clean = clean.withColumn(c, F.upper(F.trim(c)))
clean = clean.withColumn('flight_date', F.expr(
    "try_cast(concat(year, '-', lpad(cast(month as string), 2, '0'), '-', lpad(cast(day as string), 2, '0')) as date)"))
# Les codes HHMM restent disponibles ; les minutes sont dérivées de l'horaire prévu local.
clean = clean.withColumn('scheduled_departure_hhmm', F.expr('try_cast(scheduled_departure as int)'))
clean = clean.withColumn('scheduled_hour', F.floor(F.col('scheduled_departure_hhmm') / 100).cast('int'))
valid_hhmm = ((F.col('scheduled_hour').between(0, 23)) &
              ((F.col('scheduled_departure_hhmm') % 100).between(0, 59)))
valid_row = (F.col('flight_date').isNotNull() & F.col('flight_number').isNotNull() &
             F.col('cancelled').isin(0, 1) & F.col('diverted').isin(0, 1) &
             (F.col('distance') > 0) & valid_hhmm &
             F.col('airline').rlike('^[A-Z0-9]{2}$') &
             F.col('origin_airport').rlike('^[A-Z]{3}$') &
             F.col('destination_airport').rlike('^[A-Z]{3}$'))
clean = clean.withColumn('_valid_core', F.coalesce(valid_row, F.lit(False)))
REPORT['excluded_invalid_core'] = clean.filter(~F.col('_valid_core')).count()
clean = clean.filter('_valid_core').drop('_valid_core')
# Une avance n'est pas une anomalie. Les retards manquants restent NULL.
clean = (clean.withColumn('season', F.when(F.col('month').isin(12, 1, 2), 'Hiver')
          .when(F.col('month').isin(3, 4, 5), 'Printemps')
          .when(F.col('month').isin(6, 7, 8), 'Été').otherwise('Automne'))
         .withColumn('departure_slot', F.when(F.col('scheduled_hour').between(6, 11), 'Matin')
          .when(F.col('scheduled_hour').between(12, 17), 'Après-midi')
          .when(F.col('scheduled_hour').between(18, 23), 'Soir').otherwise('Nuit'))
         .withColumn('departure_delay_15', F.when((F.col('cancelled') == 0) & F.col('departure_delay').isNotNull(),
                                                (F.col('departure_delay') >= 15).cast('int')))
         .withColumn('positive_departure_delay', F.when((F.col('cancelled') == 0) & F.col('departure_delay').isNotNull(),
                                                       F.greatest(F.col('departure_delay'), F.lit(0.0)))))
count_before_join = clean.count()
REPORT['rows_before_join'] = count_before_join
assert REPORT['bronze_rows']['flights'] == (REPORT['strict_duplicates_removed'] +
       REPORT['excluded_invalid_core'] + count_before_join)

# Référentiels compacts, dont les clés ont été vérifiées uniques à l'exercice 1.1.
airlines = bronze_dfs['airlines'].select(F.upper(F.trim('iata_code')).alias('airline'),
                                        F.col('airline').alias('airline_name'))
def airport_dimension(prefix, key):
    return bronze_dfs['airports'].select(
        F.upper(F.trim('iata_code')).alias(key),
        *[F.col(c).alias(f'{prefix}_airport_name' if c == 'airport' else f'{prefix}_{c}') for c in ['airport', 'city', 'state', 'country']],
        F.expr('try_cast(latitude as double)').alias(f'{prefix}_latitude'),
        F.expr('try_cast(longitude as double)').alias(f'{prefix}_longitude'),
        F.lit(True).alias(f'{prefix}_matched'))
df_silver = (clean.join(F.broadcast(airlines), 'airline', 'left')
             .join(F.broadcast(airport_dimension('origin', 'origin_airport')), 'origin_airport', 'left')
             .join(F.broadcast(airport_dimension('destination', 'destination_airport')), 'destination_airport', 'left'))
# Indicateur combinant faits et référentiels : trajet entre deux États différents.
df_silver = df_silver.withColumn('interstate_route',
    F.when(F.col('origin_state').isNotNull() & F.col('destination_state').isNotNull(),
           F.col('origin_state') != F.col('destination_state')))
join_count = df_silver.count()
assert join_count == count_before_join, 'La jointure gauche doit conserver exactement la granularité.'
REPORT['rows_after_join'] = join_count
REPORT['join_matches'] = {}
for label, predicate, key in [
    ('airline', F.col('airline_name').isNotNull(), 'airline'),
    ('origin', F.col('origin_matched').isNotNull(), 'origin_airport'),
    ('destination', F.col('destination_matched').isNotNull(), 'destination_airport')]:
    matched = df_silver.filter(predicate).count()
    REPORT['join_matches'][label] = {'matched_rows': matched, 'total_rows': join_count,
                                    'match_rate': matched / join_count}
    print(label, ':', REPORT['join_matches'][label])
    df_silver.filter(~predicate).groupBy(key).count().orderBy(F.desc('count')).show(10, truncate=False)

print('Plan physique — rechercher les trois BroadcastHashJoin :')
df_silver.explain('simple')
plan = df_silver._jdf.queryExecution().executedPlan().toString()
assert plan.count('BroadcastHashJoin') >= 3
Path('/workspace/reports/j2_join_plan.txt').write_text(plan, encoding='utf-8')
(df_silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(SILVER_MAIN_PATH))
written = spark.read.format('delta').load(SILVER_MAIN_PATH)
assert written.count() == count_before_join
written.select('flight_date', 'airline_name', 'origin_city', 'destination_city',
               'departure_delay', 'season', 'departure_slot', 'interstate_route').show(8, truncate=False)
REPORT['silver_path'] = SILVER_MAIN_PATH
REPORT['silver_rows'] = written.count()
REPORT['silver_columns'] = len(written.columns)
REPORT['quality_checks'] = written.agg(
    F.sum(F.when(F.col('departure_delay') < 0, 1).otherwise(0)).alias('negative_delays_preserved'),
    F.sum(F.when((F.col('cancelled') == 1) & F.col('departure_delay_15').isNotNull(), 1).otherwise(0)).alias('cancelled_with_delay_flag'),
    F.sum(F.when(F.col('flight_date').isNull(), 1).otherwise(0)).alias('null_dates')).first().asDict()
assert REPORT['quality_checks']['cancelled_with_delay_flag'] == 0
assert REPORT['quality_checks']['null_dates'] == 0

# Lazy evaluation : construction sans action, puis action count déclenchant des jobs.
sc = spark.sparkContext
sc.setJobGroup('j2_lazy_demo', 'Démonstration lazy / narrow / wide')
lazy_df = written.select('airline', 'departure_delay').filter('departure_delay IS NOT NULL')
REPORT['lazy_jobs_before_action'] = len(sc.statusTracker().getJobIdsForGroup('j2_lazy_demo'))
assert REPORT['lazy_jobs_before_action'] == 0
print('Narrow : select et filter, sans redistribution de la table.')
lazy_df.explain('simple')
wide_df = lazy_df.groupBy('airline').agg(F.avg('departure_delay').alias('mean_delay'))
print('Wide : agrégation par compagnie ; Exchange hashpartitioning dans le plan.')
wide_df.explain('simple')
wide_df.count()
REPORT['lazy_jobs_after_action'] = len(sc.statusTracker().getJobIdsForGroup('j2_lazy_demo'))
assert REPORT['lazy_jobs_after_action'] > 0
sc.setLocalProperty('spark.jobGroup.id', None)
print('Jobs avant/après action :', REPORT['lazy_jobs_before_action'], REPORT['lazy_jobs_after_action'])
Path('/workspace/reports/j2_first_half.json').write_text(json.dumps(REPORT, indent=2, ensure_ascii=False), encoding='utf-8')
flights_unique.unpersist()
print('Exercices 1.1 et 2.1 terminés : table Silver validée.')

airline : {'matched_rows': 1943809, 'total_rows': 1943809, 'match_rate': 1.0}


+-------+-----+
|airline|count|
+-------+-----+
+-------+-----+



origin : {'matched_rows': 1943809, 'total_rows': 1943809, 'match_rate': 1.0}


+--------------+-----+
|origin_airport|count|
+--------------+-----+
+--------------+-----+



destination : {'matched_rows': 1943809, 'total_rows': 1943809, 'match_rate': 1.0}


+-------------------+-----+
|destination_airport|count|
+-------------------+-----+
+-------------------+-----+

Plan physique — rechercher les trois BroadcastHashJoin :


... (2 outputs left out)

Narrow : select et filter, sans redistribution de la table.


== Physical Plan ==
*(1) Filter isnotnull(departure_delay#19713)
+- *(1) ColumnarToRow
   +- FileScan parquet [airline#19704,departure_delay#19713] Batched: true, DataFilters: [isnotnull(departure_delay#19713)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[hdfs://namenode:9000/lakehouse/silver/flights], PartitionFilters: [], PushedFilters: [IsNotNull(departure_delay)], ReadSchema: struct<airline:string,departure_delay:double>


Wide : agrégation par compagnie ; Exchange hashpartitioning dans le plan.


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[airline#19704], functions=[avg(departure_delay#19713)])
   +- Exchange hashpartitioning(airline#19704, 8), ENSURE_REQUIREMENTS, [plan_id=5603]
      +- HashAggregate(keys=[airline#19704], functions=[partial_avg(departure_delay#19713)])
         +- Filter isnotnull(departure_delay#19713)
            +- FileScan parquet [airline#19704,departure_delay#19713] Batched: true, DataFilters: [isnotnull(departure_delay#19713)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[hdfs://namenode:9000/lakehouse/silver/flights], PartitionFilters: [], PushedFilters: [IsNotNull(departure_delay)], ReadSchema: struct<airline:string,departure_delay:double>




Jobs avant/après action : 0 6
Exercices 1.1 et 2.1 terminés : table Silver validée.


## Résultats et interprétation — exercices 1.1 et 2.1

**Exercices 1.1 et 2.1 exécutés le 17 novembre 2026, puis réexécutés avec des résultats identiques le 18 novembre 2026.**

- Bronze et Silver : **1 943 809 vols**. Aucun doublon strict métier, aucune exclusion selon les règles de validité appliquées, aucune erreur de conversion sur les colonnes numériques contrôlées.
- Clés de jointure renseignées ; référentiels uniques : **14 compagnies**, **322 aéroports**.
- Trois jointures gauches en **BroadcastHashJoin**, avec **100 % de rapprochement** pour chacune ; effectif strictement conservé. Un taux de rapprochement de 100 % ne garantit pas que tous les attributs enrichis sont renseignés.
- **24 756 retards de départ** et **30 794 retards d'arrivée** sont manquants dans Bronze. Ils restent manquants après typage : on ne les assimile pas à une arrivée à l'heure. Les causes de retard et d'annulation sont des champs conditionnels, à interpréter selon le statut du vol.
- Les coordonnées sont absentes pour **3 aéroports** du référentiel ; filtrer les coordonnées manquantes pour une carte et signaler cette couverture.
- **1 104 197 retards au départ négatifs** sont conservés : ils représentent des avances. Aucun vol annulé n'a d'indicateur `departure_delay_15` renseigné.
- **57 colonnes Silver**, dont date, saison, créneau du départ prévu, indicateur de retard de 15 minutes, minutes positives de retard et indicateur de trajet entre États (`interstate_route`). Ce dernier combine la table de faits et les deux référentiels d'aéroports.
- Évaluation paresseuse : **0 job** lors de la construction du DataFrame de démonstration, **6 jobs** observés après l'action. `select`/`filter` sont narrow ; le `groupBy` introduit un échange de données dans le plan.

Les règles de filtrage contrôlent date, numéro de vol, indicateurs 0/1, distance positive, codes et horaire prévu HHMM. Elles ne prouvent pas l'absence de toute anomalie métier. Les horaires locaux sont conservés sans calculer de durée entre fuseaux. La saison n'est représentée que par un mois de l'extrait : pas de conclusion annuelle.

Pour rejouer uniquement ces deux exercices, sans la section 3 :

```bash
docker exec spark-jupyter python /workspace/scripts/run_j2_first_half.py
```

Preuves : `reports/j2_first_half.json` et `reports/j2_join_plan.txt`. Le script d'exécution arrête sa session Spark après validation ; l'interface Spark UI n'est disponible que pendant une session active.

## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [4]:
# Section 3 autonome : elle relit Silver sur HDFS et peut être rejouée sans refaire les jointures.
import json
from pathlib import Path

from delta.tables import DeltaTable
from pyspark.sql import functions as F

SILVER_MAIN_PATH = '/lakehouse/silver/flights'
DELTA_REPORT = {'scope': 'Exercices 3.1 et 3.2', 'silver_path': SILVER_MAIN_PATH}
silver_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
version_before = silver_delta.history(1).first()['version']
rows_before = spark.read.format('delta').load(SILVER_MAIN_PATH).count()
DELTA_REPORT['version_before_tests'] = version_before
DELTA_REPORT['rows_before_tests'] = rows_before

# 1. Journal transactionnel : un fichier JSON par commit dans _delta_log.
log_dir = f'{SILVER_MAIN_PATH}/_delta_log'
log0_path = f'{log_dir}/00000000000000000000.json'
spark.read.text(log0_path).show(8, truncate=110)

jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
commit_files = sorted(s.getPath().getName() for s in fs.listStatus(jvm.org.apache.hadoop.fs.Path(log_dir))
                      if s.getPath().getName().endswith('.json'))
print('Commits présents dans _delta_log :', commit_files)

# Chaque ligne du commit est une action : commitInfo, protocol, metaData, add (fichier Parquet ajouté)...
log0 = spark.read.json(log0_path)
actions = {a: log0.filter(F.col(a).isNotNull()).count() for a in log0.columns}
commit_info = log0.filter('commitInfo IS NOT NULL').select('commitInfo.*').first().asDict(recursive=True)
schema_fields = json.loads(log0.filter('metaData IS NOT NULL').select('metaData.schemaString').first()[0])['fields']
DELTA_REPORT['log_commit_0'] = {
    'actions': actions,
    'operation': commit_info.get('operation'),
    'mode': (commit_info.get('operationParameters') or {}).get('mode'),
    'num_files': (commit_info.get('operationMetrics') or {}).get('numFiles'),
    'num_output_rows': (commit_info.get('operationMetrics') or {}).get('numOutputRows'),
    'schema_fields': len(schema_fields),
}
DELTA_REPORT['commit_files_before_tests'] = commit_files
print('Commit 0 :', json.dumps(DELTA_REPORT['log_commit_0'], ensure_ascii=False))

# 2. Schema Enforcement : un DataFrame aux colonnes inconnues doit être refusé, sans créer de version.
bad_df = spark.createDataFrame([('test_1', 'colonne_intruse')], ['fake_id', 'intruder_col'])
rejection = None
try:
    bad_df.write.format('delta').mode('append').save(SILVER_MAIN_PATH)
except Exception as e:
    rejection = f'{type(e).__name__}: {str(e).splitlines()[0]}'
assert rejection is not None, 'Delta aurait dû refuser ce schéma incompatible.'
print('Schema Enforcement validé — écriture invalide bloquée :', rejection)
version_after_reject = silver_delta.history(1).first()['version']
rows_after_reject = spark.read.format('delta').load(SILVER_MAIN_PATH).count()
assert version_after_reject == version_before and rows_after_reject == rows_before
DELTA_REPORT['schema_enforcement'] = {'rejected': True, 'error': rejection,
                                      'version_unchanged': True, 'rows_unchanged': True}
print('Aucune nouvelle version ni ligne : la transaction refusée n\'a rien écrit.')

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794922717862,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"f2eb9c53-c21c-4ca3-b2e0-2069dae21e0a","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-43b99441-5759-4ff5-bc4f-7ce4920ad977-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-24c26f67-9dd5-4bd1-9229-bec648d60a3a-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-53d49903-23d1-4210-862f-03b8dddde74c-c000.snappy.parquet","partition

Commit 0 : {"actions": {"add": 8, "commitInfo": 1, "metaData": 1, "protocol": 1}, "operation": "WRITE", "mode": "Overwrite", "num_files": "8", "num_output_rows": "1943809", "schema_fields": 57}


Schema Enforcement validé — écriture invalide bloquée : AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.


Aucune nouvelle version ni ligne : la transaction refusée n'a rien écrit.


### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

In [5]:
# Clé métier d'un vol ; l'opérateur <=> compare aussi correctement les NULL.
KEY = ['flight_date', 'airline', 'flight_number', 'origin_airport', 'scheduled_departure']
silver = spark.read.format('delta').load(SILVER_MAIN_PATH)
DEMO_TAG = 'demo_merge_j2'

# Lot correctif simulé : deux vols retardés de 10 à 14 min reçoivent +5 min, ce qui fait basculer leur indicateur 15 min.
to_fix = (silver.filter('cancelled = 0 AND departure_delay BETWEEN 10 AND 14')
          .orderBy(*KEY).limit(2))
corrections = (to_fix.withColumn('departure_delay', F.col('departure_delay') + 5)
               .withColumn('departure_delay_15', (F.col('departure_delay') >= 15).cast('int'))
               .withColumn('positive_departure_delay', F.greatest('departure_delay', F.lit(0.0))))
# Nouveau vol fictif : copie d'un vol existant avec un numéro inexistant, donc non apparié.
assert silver.filter('flight_number = 99999').count() == 0
new_flight = silver.filter("airline = 'AA'").orderBy(*KEY).limit(1).withColumn('flight_number', F.lit(99999).cast('int'))
source = (corrections.unionByName(new_flight)
          .withColumn('_source_dataset', F.lit(DEMO_TAG))
          .withColumn('_ingested_at', F.current_timestamp()))
# Le lot est figé sur le driver (3 lignes) pour ne pas dépendre d'une relecture de la cible pendant le MERGE.
source = spark.createDataFrame(source.collect(), source.schema)
match_cond = ' AND '.join(f't.{k} <=> s.{k}' for k in KEY)
assert silver.alias('t').join(source.alias('s'), F.expr(match_cond)).count() == 2, 'Chaque correction doit viser un seul vol.'

updated_cols = ['departure_delay', 'departure_delay_15', 'positive_departure_delay', '_source_dataset', '_ingested_at']
try:
    (silver_delta.alias('t').merge(source.alias('s'), match_cond)
     .whenMatchedUpdate(set={c: f's.{c}' for c in updated_cols})
     .whenNotMatchedInsertAll()
     .execute())
    
    merge_commit = silver_delta.history(1).first()
    version_merge = merge_commit['version']
    merge_metrics = merge_commit['operationMetrics']
    print('MERGE : version', version_merge, '| mis à jour :', merge_metrics['numTargetRowsUpdated'],
          '| insérés :', merge_metrics['numTargetRowsInserted'])
    assert merge_commit['operation'] == 'MERGE'
    assert int(merge_metrics['numTargetRowsUpdated']) == 2 and int(merge_metrics['numTargetRowsInserted']) == 1
    
    # Time Travel : comparer la version précédant le MERGE et la version courante sur les vols concernés.
    before_df = spark.read.format('delta').option('versionAsOf', version_before).load(SILVER_MAIN_PATH)
    after_df = spark.read.format('delta').option('versionAsOf', version_merge).load(SILVER_MAIN_PATH)
    rows_version_0 = spark.read.format('delta').option('versionAsOf', 0).load(SILVER_MAIN_PATH).count()
    rows_after_merge = after_df.count()
    assert rows_after_merge == rows_before + 1
    cols = KEY + ['departure_delay', 'departure_delay_15', '_source_dataset']
    keys_df = source.select(*KEY)
    comparison = (keys_df.alias('k')
                  .join(before_df.select(*cols).alias('avant'), F.expr(' AND '.join(f'k.{k} <=> avant.{k}' for k in KEY)), 'left')
                  .join(after_df.select(*cols).alias('apres'), F.expr(' AND '.join(f'k.{k} <=> apres.{k}' for k in KEY)), 'left')
                  .select(*[F.col(f'k.{k}') for k in ['flight_date', 'airline', 'flight_number', 'origin_airport']],
                          F.col('avant.departure_delay').alias('retard_avant'), F.col('apres.departure_delay').alias('retard_apres'),
                          F.col('avant.departure_delay_15').alias('flag15_avant'), F.col('apres.departure_delay_15').alias('flag15_apres'),
                          F.col('apres._source_dataset').alias('source_apres')))
    comparison_rows = comparison.orderBy('flight_number').collect()  # figé avant le RESTORE
    comparison.orderBy('flight_number').show(truncate=False)
    print(f'Lignes : version 0 = {rows_version_0:,} | avant MERGE (v{version_before}) = {rows_before:,} '
          f'| après MERGE (v{version_merge}) = {rows_after_merge:,}')
    
finally:
    # Rétablir Silver même si un contrôle de la démonstration échoue.
    spark.sql(f'RESTORE TABLE delta.`{SILVER_MAIN_PATH}` TO VERSION AS OF {version_before}')

restored = spark.read.format('delta').load(SILVER_MAIN_PATH)
rows_restored = restored.count()
demo_rows_left = restored.filter(F.col('_source_dataset') == DEMO_TAG).count()
assert rows_restored == rows_before and demo_rows_left == 0

history_df = spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_MAIN_PATH}`")
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
assert history_df.count() >= 2
last_ops = [r['operation'] for r in history_df.orderBy(F.desc('version')).limit(2).collect()]
assert last_ops == ['RESTORE', 'MERGE'], last_ops

DELTA_REPORT['merge'] = {
    'match_key': KEY, 'version': version_merge,
    'rows_updated': int(merge_metrics['numTargetRowsUpdated']),
    'rows_inserted': int(merge_metrics['numTargetRowsInserted']),
    'comparison': [r.asDict() for r in comparison_rows],
}
DELTA_REPORT['time_travel'] = {'rows_version_0': rows_version_0, 'rows_before_merge': rows_before,
                               'rows_after_merge': rows_after_merge}
DELTA_REPORT['restore'] = {'to_version': version_before, 'version': version_merge + 1,
                           'rows_after_restore': rows_restored, 'demo_rows_left': demo_rows_left}
DELTA_REPORT['history_last_operations'] = last_ops
Path('/workspace/reports/j2_second_half.json').write_text(
    json.dumps(DELTA_REPORT, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Exercices 3.1 et 3.2 validés ; rapport : reports/j2_second_half.json')

MERGE : version 2 | mis à jour : 2 | insérés : 1


+-----------+-------+-------------+--------------+------------+------------+------------+------------+-------------+
|flight_date|airline|flight_number|origin_airport|retard_avant|retard_apres|flag15_avant|flag15_apres|source_apres |
+-----------+-------+-------------+--------------+------------+------------+------------+------------+-------------+
|2015-01-01 |AA     |65           |JFK           |10.0        |15.0        |0           |1           |demo_merge_j2|
|2015-01-01 |AA     |96           |DFW           |11.0        |16.0        |0           |1           |demo_merge_j2|
|2015-01-01 |AA     |99999        |JFK           |NULL        |-5.0        |NULL        |0           |demo_merge_j2|
+-----------+-------+-------------+--------------+------------+------------+------------+------------+-------------+

Lignes : version 0 = 1,943,809 | avant MERGE (v1) = 1,943,809 | après MERGE (v2) = 1,943,810


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      3|2026-11-19 10:35:38.211|  RESTORE|{numRestoredFiles -> 2, removedFilesSize -> 19185488, numRemovedFiles -> 3, r...|
|      2|2026-11-19 10:35:02.854|    MERGE|{numTargetRowsCopied -> 485613, numTargetRowsDeleted -> 0, numTargetBytesRemo...|
|      1|2026-11-19 10:34:24.437|    WRITE|{numFiles -> 8, numRemovedFiles -> 8, numRemovedBytes -> 75978889, numOutputR...|
|      0|2026-11-17 13:38:38.072|    WRITE|{numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> ...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+


## Résultats et interprétation — exercices 3.1 et 3.2

**Exécutés le 18 novembre 2026.** Preuves : `reports/j2_second_half.json`.

**Journal `_delta_log`.** Chaque transaction validée produit un fichier JSON numéroté. Le commit 0 contient 11 actions : `commitInfo` (opération `WRITE`, mode `Overwrite`, 1 943 809 lignes), `protocol`, `metaData` (schéma de 57 colonnes) et **8 actions `add`**, une par fichier Parquet écrit. Lire une version consiste à rejouer ces actions : la table est l'ensemble des fichiers ajoutés et non retirés.

**Schema Enforcement.** L'ajout d'un DataFrame aux colonnes `fake_id` et `intruder_col` est refusé par `DELTA_METADATA_MISMATCH`. Aucune version n'est créée et l'effectif reste à 1 943 809 : la transaction échoue en entier, sans écriture partielle. L'évolution du schéma resterait possible, mais seulement de façon explicite (`mergeSchema`).

**MERGE.** Clé de rapprochement : date, compagnie, numéro de vol, aéroport d'origine et départ prévu, avec `<=>` pour comparer aussi les valeurs nulles. Le lot simulé contenait deux corrections et un vol nouveau :

| Vol | Retard avant | Retard après | Indicateur ≥ 15 min avant → après |
|---|---:|---:|---|
| AA 65, JFK, 01/01/2015 | 10 | 15 | 0 → 1 |
| AA 96, DFW, 01/01/2015 | 11 | 16 | 0 → 1 |
| AA 99999 (fictif) | — | −5 | inséré |

Résultat : **2 lignes mises à jour, 1 insérée**. Les colonnes dérivées (`departure_delay_15`, `positive_departure_delay`) sont recalculées dans le lot et le lignage indique `demo_merge_j2`. Delta fonctionne en *copy-on-write* : les fichiers Parquet contenant les vols visés sont réécrits (485 613 lignes recopiées) plutôt que modifiés sur place.

**Time Travel.** La version précédant le MERGE compte 1 943 809 lignes, la version du MERGE 1 943 810. Les valeurs avant/après ci-dessus sont lues avec `versionAsOf` sur la même table. La version 0 reste lisible (1 943 809 lignes).

**Remise en état.** Le lot étant une démonstration, `RESTORE ... TO VERSION AS OF` rétablit l'état validé : 1 943 809 lignes et aucune ligne `demo_merge_j2`. L'historique conserve `WRITE`, `MERGE` puis `RESTORE` : la correction reste auditable sans fausser les analyses Gold. Les versions 1 et 2 de l'historique proviennent d'un essai préalable identique ; la réécriture de la section 2 a créé la version 3.

In [6]:
spark.stop()